# Mean-field approaches

Companion notebook to Chapter 6 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here.  The code is
the same as in `BookManybody/BookMaterial/Programs/hartreefock.py`.

Contents:

1. The self-consistent field loop, and Brillouin's theorem
2. When the mean field does nothing: the pairing model
3. Thouless' theorem, verified in the determinant basis
4. Stability, and the Lipkin instability at $\chi = 1$
5. The Baker-Campbell-Hausdorff expansion
6. The Trotter-Suzuki splitting
7. The infinite homogeneous electron gas

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import hartreefock as hf

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. The self-consistent field loop

The Hartree-Fock equations

$$\sum_{\beta} h^{\rm HF}_{\alpha\beta} C_{i\beta}
  = \varepsilon_i^{\rm HF} C_{i\alpha},
\qquad
h^{\rm HF}_{\alpha\beta} = \langle\alpha|\hat h_0|\beta\rangle
  + \sum_{\gamma\delta}\rho_{\gamma\delta}
    \langle\alpha\gamma|\hat v|\beta\delta\rangle_{\rm AS}$$

are nonlinear only because $\rho$ depends on the eigenvectors we are looking
for.  Each iteration is therefore a single Hermitian diagonalisation -- the
algorithms of Chapter 1 -- wrapped in a fixed-point loop.

In [ ]:
hf.demo_scf()

### Watching it converge

The energy falls monotonically and the single-particle energies settle.  The
density matrix is the projector onto the occupied subspace, so it is the
object that actually converges; the individual orbitals are fixed only up to
a unitary mixing among themselves.

In [ ]:
h0, v = hf.trap_system(n_orbitals=8)
scf = hf.SelfConsistentField(h0, v, n_particles=4)
scf.run(verbose=True)

rho = scf.density(scf.C)
print("\ntrace of rho          :", np.trace(rho))
print("|rho^2 - rho|         :", np.abs(rho @ rho - rho).max())
print("eigenvalues of rho    :", np.linalg.eigvalsh(rho))
print("\nmax |f_ai| (Brillouin):", f"{scf.brillouin():.2e}")

In [ ]:
it, energies, drifts = zip(*scf.history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(it, energies, "o-")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel(r"$E^{\rm HF}$")
ax[0].set_title("energy")
ax[1].semilogy(it[1:], drifts[1:], "o-")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel(r"mean $|\Delta\varepsilon|$")
ax[1].set_title("convergence")
fig.tight_layout()
plt.show()

## 2. When the mean field does nothing

The pairing interaction moves complete pairs and conserves the seniority, so
it has no matrix element between the reference and a $1p$-$1h$ state.  The
Fock matrix is diagonal from the first iteration and Hartree-Fock gains
exactly nothing -- the whole correlation energy has to come from the doubles,
as Chapter 5 found.

In [ ]:
hf.demo_pairing()

## 3. Thouless' theorem

$$|c'\rangle = \exp\Big\{\sum_{a>F}\sum_{i\le F} C_{ai}
  \hat a^\dagger_a \hat a_i\Big\}|c\rangle$$

is again a *single* Slater determinant, built from
$\hat b^\dagger_i = \hat a^\dagger_i + \sum_a C_{ai}\hat a^\dagger_a$.  The
reason is that the one-particle-one-hole operators commute and each squares
to zero, so the exponential collapses to a finite product.

In [ ]:
hf.demo_thouless()

Note the distinction that matters for the stability analysis: $e^{\hat T}$ is
the product $\prod_i (1 + \hat T_i)$, **not** $1 + \hat T$.  The cross terms
$\hat T_i \hat T_j$ with $i \neq j$ are $2p$-$2h$ admixtures, second order in
the amplitudes.  Keeping only $1p$-$1h$ is exact for an infinitesimal
variation, which is all the Hartree-Fock condition needs.

## 4. Stability of the Hartree-Fock solution

Going to second order in the amplitudes gives

$$\Delta E = \tfrac12 \langle \chi | \hat M | \chi\rangle,
\qquad
\hat M = \begin{pmatrix} \Delta + A & B \\ B^* & \Delta + A^*\end{pmatrix},$$

and the solution is a local minimum only if $\hat M$ is positive
semi-definite.  $\hat M$ is the RPA matrix, so "stable Hartree-Fock" and
"real RPA frequencies" are the same statement.

The Lipkin model shows the transition explicitly at
$\chi = |V|(N-1)/\varepsilon = 1$.

In [ ]:
hf.demo_stability()

In [ ]:
# the mean-field energy surface on either side of the transition
alpha = np.linspace(-np.pi, np.pi, 400)
fig, ax = plt.subplots(figsize=(7, 4))
for chi in (0.5, 1.0, 1.5, 2.5):
    model = hf.LipkinHF(N=4, eps=1.0, V=chi / 3.0)
    ax.plot(alpha, model.mean_field_energy(alpha) / model.N,
            label=rf"$\chi = {chi}$")
    a_min, e_min, deformed = model.mean_field_minimum()
    if deformed:
        ax.plot([a_min, -a_min], [e_min / model.N] * 2, "k.", ms=9)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$E(\alpha)/N$")
ax.set_title("Lipkin mean-field energy surface")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
# the lowest eigenvalue of M crosses zero exactly at chi = 1
chis = np.linspace(0.05, 2.5, 40)
lowest = []
for chi in chis:
    model = hf.LipkinHF(N=4, eps=1.0, V=chi / 3.0)
    S = hf.StabilityMatrix(model.matrix(), model.states, model.index,
                           model.reference, model.N, model.n)
    lowest.append(S.eigenvalues()[0])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(chis, lowest, "o-", label=r"$\lambda_{\min}(\hat M)$")
ax.plot(chis, 1.0 - chis, "--", label=r"$\varepsilon(1-\chi)$")
ax.axhline(0.0, color="k", lw=0.8)
ax.axvline(1.0, color="k", lw=0.8, ls=":")
ax.set_xlabel(r"$\chi$"); ax.set_ylabel("lowest eigenvalue")
ax.legend(); fig.tight_layout(); plt.show()

## 5. The Baker-Campbell-Hausdorff expansion

$$Z = \log(e^X e^Y) = X + Y + \tfrac12[X,Y]
  + \tfrac1{12}\big([X,[X,Y]]+[Y,[Y,X]]\big)
  - \tfrac1{24}\big[Y,[X,[X,Y]]\big] + \cdots$$

Every term beyond the first is a nested commutator, which is why $Z$ stays in
the same Lie algebra as $X$ and $Y$.  In Thouless' theorem all the
commutators vanish and the series stops at the first term.

In [ ]:
hf.demo_bch()

In [ ]:
# compare the truncated series against the exact logarithm
A = np.array([[0.0, 0.1], [0.0, 0.0]])
B = np.array([[0.0, 0.0], [0.1, 0.0]])
Z_exact = hf.bch_exact_log(A, B).real
print("exact  Z = log(e^A e^B):\n", Z_exact)
for n in (1, 2, 3, 4):
    Z = hf.bch_series(A, B, n)
    print(f"order {n}:  ||Z_n - Z|| = {np.linalg.norm(Z - Z_exact):.3e}")

## 6. The Trotter-Suzuki splitting

Reading the BCH formula backwards gives the splitting error:

$$e^{-iH_1\Delta}e^{-iH_2\Delta}
 = \exp\!\big(-i(H_1+H_2)\Delta - \tfrac12[H_1,H_2]\Delta^2 + O(\Delta^3)\big).$$

The symmetric (Strang) product removes the leading commutator, leaving a
local error $O(\Delta^3)$.

In [ ]:
hf.demo_trotter()

In [ ]:
X = np.array([[0.0, 1.0], [1.0, 0.0]])
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
rows = hf.trotter_error(X, Z, t=1.0, steps=(1, 2, 4, 8, 16, 32, 64, 128, 256))
N, e1, e2 = np.array(rows).T

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(N, e1, "o-", label="first order")
ax.loglog(N, e2, "s-", label="second order")
ax.loglog(N, e1[0] / N, "k--", lw=0.8, label=r"$\propto 1/N$")
ax.loglog(N, e2[0] / N**2, "k:", lw=0.8, label=r"$\propto 1/N^2$")
ax.set_xlabel("number of Trotter steps $N$")
ax.set_ylabel(r"$\|U_{\rm trot} - U_{\rm exact}\|$")
ax.legend(); fig.tight_layout(); plt.show()

## 7. The infinite homogeneous electron gas

With $x = k/k_F$ and

$$F(x) = \frac12 + \frac{1-x^2}{4x}\ln\left|\frac{1+x}{1-x}\right|,$$

the Hartree-Fock single-particle energy is

$$\frac{\varepsilon_k^{\rm HF}}{\varepsilon_0^F}
  = x^2 - 0.663\,r_s\,F(x).$$

In [ ]:
hf.demo_electron_gas()

In [ ]:
gas = hf.ElectronGas(rs_over_a0=4.0)
x = np.linspace(1e-6, 2.0, 800)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, gas.energy(x), label=r"Hartree-Fock, $r_s/a_0 = 4$")
ax.plot(x, x**2, label="free electrons")
ax.axvline(1.0, color="k", lw=0.8, ls=":")
ax.set_xlabel(r"$k/k_F$")
ax.set_ylabel(r"$\varepsilon_k / \varepsilon_0^F$")
ax.set_title("single-particle energy")
ax.legend(); fig.tight_layout(); plt.show()

print(f"band width, Hartree-Fock : {gas.band_width():.4f}")
print(f"band width, free gas     : 1.0000")
print("Exchange more than doubles the width of the occupied band.")

### The pathology at the Fermi surface

$F'(x)$ diverges logarithmically at $x=1$, so the slope of
$\varepsilon^{\rm HF}_k$ is infinite there, the effective mass goes to zero
and the level density

$$n(\varepsilon) = \frac{\Omega k^2}{2\pi^2}
  \left(\frac{\partial\varepsilon}{\partial k}\right)^{-1}$$

vanishes at the Fermi surface.  Real metals have a finite density of states
-- it is what gives them a linear specific heat.  The culprit is the
unscreened $1/q^2$ of the bare Coulomb interaction, and the cure is the
random-phase approximation.

In [ ]:
deltas = np.logspace(-1, -8, 30)
mstar = [gas.effective_mass(d) for d in deltas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(deltas, mstar, "o-")
ax.invert_xaxis()
ax.set_xlabel(r"$1 - k/k_F$")
ax.set_ylabel(r"$m^*_{\rm HF} / m$")
ax.set_title("the Hartree-Fock effective mass vanishes at the Fermi surface")
fig.tight_layout(); plt.show()

### The energy per electron

$$\frac{E_0}{N} = \frac{e^2}{2a_0}
  \left[\frac{2.21}{r_s^2} - \frac{0.916}{r_s}\right]$$

Kinetic repulsion $\propto r_s^{-2}$ against exchange attraction
$\propto r_s^{-1}$: the sum has a minimum, so the gas is bound already at the
Hartree-Fock level, at a density in the range of the alkali metals.

In [ ]:
rs = np.linspace(1.0, 12.0, 400)
E = hf.ElectronGas.energy_per_electron(rs)
rs_eq, e_eq = hf.ElectronGas.equilibrium()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(rs, E, label=r"$E_0/N$")
ax.plot(rs, 2.21 / rs**2, "--", lw=0.9, label="kinetic")
ax.plot(rs, -0.916 / rs, "--", lw=0.9, label="exchange")
ax.plot(rs_eq, e_eq, "ko", ms=7)
ax.axhline(0.0, color="k", lw=0.8)
ax.set_ylim(-0.3, 0.6)
ax.set_xlabel(r"$r_s = r_0/a_0$"); ax.set_ylabel("energy per electron [Ry]")
ax.legend(); fig.tight_layout(); plt.show()

print(f"equilibrium at r_s = {rs_eq:.4f}")
print(f"E_0/N = {e_eq:.6f} Ry = {e_eq * 13.6057:.4f} eV")
print("For comparison, the correlation energy near equilibrium is about")
print("-0.06 Ry, so exchange alone accounts for roughly half the binding.")

## The full program

Everything above lives in
`BookManybody/BookMaterial/Programs/hartreefock.py`, which runs as a script
and prints all seven demonstrations of the chapter.

In [ ]:
print(open(hf.__file__).read())